In [1]:
import h5py
import numpy as np
import pandas as pd
import json
import os

h5_path = "../data/raw/TheCycloneImageDataset/Cyclone_Images.h5"
labels_path = "../data/raw/TheCycloneImageDataset/Cyclone_Labels h5.npy"

labels = np.load(labels_path, allow_pickle=True)
wind_speeds = labels[:, 5].astype(float)

with h5py.File(h5_path, 'r') as f:
    total_images = f['Images'].shape[0]

print("Total images:", total_images)

Total images: 21076


In [2]:
def wind_to_category(speed):
    if speed < 34: return "Depression"
    elif speed < 48: return "Deep Depression"
    elif speed < 64: return "Cyclonic Storm"
    elif speed < 90: return "Severe Cyclonic Storm"
    elif speed < 120: return "Very Severe Cyclonic Storm"
    else: return "Extremely Severe / Super Cyclone"

categories = np.array([wind_to_category(s) for s in wind_speeds])

# Encode categories to integers (needed for model training)
unique_categories = sorted(set(categories))
category_to_int = {cat: i for i, cat in enumerate(unique_categories)}
int_to_category = {i: cat for cat, i in category_to_int.items()}

encoded_labels = np.array([category_to_int[c] for c in categories])

print("Category mapping:", category_to_int)

Category mapping: {np.str_('Cyclonic Storm'): 0, np.str_('Deep Depression'): 1, np.str_('Depression'): 2, np.str_('Extremely Severe / Super Cyclone'): 3, np.str_('Severe Cyclonic Storm'): 4, np.str_('Very Severe Cyclonic Storm'): 5}


In [3]:
from sklearn.model_selection import train_test_split

indices = np.arange(total_images)

# First split: 70% train, 30% temp
train_idx, temp_idx = train_test_split(
    indices, test_size=0.30, stratify=encoded_labels, random_state=42
)

# Second split: temp -> 15% val, 15% test
temp_labels = encoded_labels[temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=temp_labels, random_state=42
)

print("Train:", len(train_idx), "Val:", len(val_idx), "Test:", len(test_idx))

Train: 14753 Val: 3161 Test: 3162


In [4]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(encoded_labels[train_idx]),
    y=encoded_labels[train_idx]
)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}
print("Class weights:", class_weight_dict)

Class weights: {0: np.float64(1.0908754806270333), 1: np.float64(0.7655147364051473), 2: np.float64(0.4600249454318678), 3: np.float64(4.077667219458264), 4: np.float64(1.3155876582842876), 5: np.float64(1.6726757369614513)}


In [5]:
os.makedirs("../data/processed", exist_ok=True)

np.save("../data/processed/train_idx.npy", train_idx)
np.save("../data/processed/val_idx.npy", val_idx)
np.save("../data/processed/test_idx.npy", test_idx)
np.save("../data/processed/encoded_labels.npy", encoded_labels)

with open("../data/processed/category_mapping.json", "w") as f:
    json.dump({"category_to_int": category_to_int, "int_to_category": int_to_category}, f, indent=2)

with open("../data/processed/class_weights.json", "w") as f:
    json.dump({str(k): float(v) for k, v in class_weight_dict.items()}, f, indent=2)

print("All preprocessing artifacts saved to data/processed/")

All preprocessing artifacts saved to data/processed/
